In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gamma
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression


expeditions = "/Users/mostafazamaniturk/Documents/AAI_500/AAI-500-project/data/expeditions.csv"

# Load the CSV file
df = pd.read_csv(expeditions)

df = df[df['max_elev_reached'] > 7000]

#print(df[df['max_elev_reached'] > 7000]['max_elev_reached'])

#print(df[df['mbrs_summited'] > 0]['mbrs_summited'])
#print(len(df[df['mbrs_summited'] > 0]['mbrs_summited']))
#print(len(df[df['mbrs_summited'] == 0]['mbrs_summited']))
#print(len(df['mbrs_summited']))

features = ['year', 'season', 'max_elev_reached', 'is_o2_used', 'members', 'agency']

df = df[features + ['mbrs_summited']]

# Convert target to binary
df['success'] = df['mbrs_summited'].apply(lambda x: 1 if x > 0 else 0)

# Convert categorical to numeric
df = pd.get_dummies(df, columns=['season', 'is_o2_used', 'agency'], drop_first=True)

# Drop original 'mbrs_summited'
df = df.drop(columns=['mbrs_summited'])

# Drop missing values
df = df.dropna()

X = df.drop('success', axis=1)
y = df['success']

# Make sure all numeric
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

#print(confusion_matrix(y_test, y_pred))
#print(classification_report(y_test, y_pred))

coeffs = pd.DataFrame({
    'Feature': X.columns,
   'Coefficient': model.coef_[0],
    'Odds Ratio': np.exp(model.coef_[0])
}).sort_values(by='Odds Ratio', ascending=False)

print(coeffs)

success_rate = df['success'].mean()
print(f"Success rate: {success_rate:.2%}")

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


                               Feature  Coefficient  Odds Ratio
134             agency_Everest Parivar     1.943645    6.984161
469       agency_Summit Nepal Trekking     1.106282    3.023098
337             agency_Multi Adventure     1.105389    3.020399
9    agency_Active Holiday Nepal Treks     0.940906    2.562302
538              agency_Yeti Adventure     0.933062    2.542282
..                                 ...          ...         ...
288                       agency_Kunga    -1.167276    0.311214
5                        season_Winter    -1.172566    0.309572
347                 agency_Nepal Himal    -1.303024    0.271709
81            agency_Bochi Bochi Treks    -1.321627    0.266701
4                        season_Summer    -2.122988    0.119673

[542 rows x 3 columns]
Success rate: 68.02%
[[152 212]
 [121 704]]
              precision    recall  f1-score   support

           0       0.56      0.42      0.48       364
           1       0.77      0.85      0.81       825



/Users/mostafazamaniturk/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Interpreting model:

The goal is to predict whether an expedition above 7000m was successful, meaning at least one member reached the summit (success = 1) vs. no one made it (success = 0).

The "mbrs_summited" column has the binary numbers, so the linear regression is not working here. 

Model performance


Confusion Matrix

|                          | Predicted No Summit  | Predicted Summit     |
| ------------------------ | -------------------- | -------------------- |
| **Actual No Summit (0)** | 152 (True Negatives) | 212 (False Positives) |
| **Actual Summit (1)**    | 121 (False Negatives) | 704 (True Positives) |


Accuracy: 77%

Precision for success (1): 72%, when model predicts success, it is right 72% of the time.

Recall for success: 91%, I catches 91% of all actual successful expeditions.

F1-Score: 81%, The balance between precision and recall need to be checked.

It can be a good performance for a simple logistic regression.


Feature importance (odds Ratios)

| Feature                 | Coefficient | Odds Ratio |
| ----------------------- | ----------- | ---------- |
| agency\_Himalaya Exped. | ...         | ....       |
| is\_o2\_used\_True      | ....        | .....      |
| season\_Spring          | ....        | ....       |
| members                 | ....        | .....      |
| max\_elev\_reached      | ....        | ....       |
| agency\_Unknown         | .....       | .....      |


The odds ratio > 1, increases likelihood of a successful climb.
The odds ration < 1, decrease likelihood.

Final summary

| Section                       | Key Insight                                                                       |
| ----------------------------- | --------------------------------------------------------------------------------- |
| **Accuracy**                  | 77% – model predicts almost well                                                         |
| **Most Important Predictors** | Oxygen use, Spring season, Reliable agency                                        |
| **Danger Signals**            | Unknown agency, no oxygen, off-season                                             |
| **Practical Tip**             | For expedition planning: go with reliable agency, in spring, and with oxygen gear |
